# exp_14 analysis - partial network + ERA5-vs-IMD characterization

**Run all cells top-to-bottom** - figures render inline AND save to results/. Both use the TRUE run numbers.
(Setup cell sets `%matplotlib inline`; plot cells end with `plt.show()` so they display in the output cells.)

In [ ]:
%matplotlib inline
import numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import LineString

ROOT = "C:/Users/rishe/Dissertation"
RES  = ROOT + "/experiments/results/exp_14_gat_gru_imd_minimap"
SHP  = ROOT + "/data/West_Bengal/District_shape_West_Bengal.shp"

sel = pd.read_csv(RES + "/selected_stations.csv")
selected = sel.station_id.tolist()
coords = pd.read_csv(ROOT + "/data/wb_station_coords.csv").rename(columns={"latitude":"lat","longitude":"lon"})
dates = pd.date_range("2000-01-01","2020-12-31",freq="D")
imd = (pd.read_parquet(ROOT + "/data/preprocessed_rain_data.parquet", columns=["station_id","date","rainfall"])
         .pivot_table(index="date",columns="station_id",values="rainfall",aggfunc="first")
         .reindex(dates)[selected].ffill().bfill().fillna(0))
era5_full_df = pd.read_parquet(ROOT + "/data/era5_pivot_data/rain_pivot.parquet")
era5 = era5_full_df.reindex(dates)[selected]

def haversine(la1,lo1,la2,lo2):
    R=6371; la1,lo1,la2,lo2=map(np.radians,[la1,lo1,la2,lo2]); dlat=la2-la1; dlon=lo2-lo1
    return 2*R*np.arcsin(np.sqrt(np.sin(dlat/2)**2+np.cos(la1)*np.cos(la2)*np.sin(dlon/2)**2))

### A - Partial-network coverage map

In [ ]:
# partial-network coverage map
wb = gpd.read_file(SHP)
if wb.crs is None: wb = wb.set_crs(4326)
wb = wb.to_crs(4326)
era5_names = era5_full_df.columns.tolist()
coords_full = coords[coords.station_id.isin(era5_names)].copy()
full_pts = gpd.GeoDataFrame(coords_full, geometry=gpd.points_from_xy(coords_full.lon, coords_full.lat), crs=4326)
sel_pts  = gpd.GeoDataFrame(sel, geometry=gpd.points_from_xy(sel.lon, sel.lat), crs=4326)
la = sel.lat.values; lo = sel.lon.values; lines=[]
for i in range(len(sel)):
    for j in range(i+1,len(sel)):
        if haversine(la[i],lo[i],la[j],lo[j])<=150:
            lines.append(LineString([(lo[i],la[i]),(lo[j],la[j])]))
edges = gpd.GeoDataFrame(geometry=lines, crs=4326)

fig, ax = plt.subplots(figsize=(8.5,9))
wb.boundary.plot(ax=ax, color="0.6", linewidth=0.7, zorder=1)
full_pts.plot(ax=ax, color="0.45", markersize=14, alpha=0.55, zorder=2, label="Full ERA5 graph ("+str(len(coords_full))+" stations)")
if len(edges): edges.plot(ax=ax, color="tab:blue", linewidth=0.9, alpha=0.55, zorder=3)
sel_pts.plot(ax=ax, color="tab:blue", marker="*", markersize=230, edgecolor="k", linewidth=0.4, zorder=4, label="exp_14 subset ("+str(len(sel))+" stations)")
ax.set_title("Partial-network coverage: exp_14 IMD minimap vs full ERA5 graph\n(150 km edges shown for the subset)", fontsize=12)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.legend(loc="upper left", fontsize=10, framealpha=0.9)
cap = ("Subset = "+str(len(sel))+"/"+str(len(coords_full))+" nodes ("+("%.0f"%(100*len(sel)/len(coords_full)))+"%). "
       "A sparser graph carries less spatial context, so absolute RMSE is NOT comparable to full-graph ERA5 runs;\n"
       "comparisons in this study are made WITHIN exp_14 (IMD vs ERA5 on the same 16 nodes).")
plt.figtext(0.5, 0.005, cap, ha="center", va="bottom", fontsize=8.5, style="italic", color="0.25")
plt.tight_layout(rect=(0,0.04,1,1)); plt.savefig(RES + "/partial_network_map.png", dpi=130); plt.show()
print("WROTE partial_network_map.png | edges:", len(edges), "| full pts:", len(coords_full))

### B - ERA5 vs IMD characterization

In [ ]:
# ERA5 vs IMD characterization
mon = np.array(imd.index.month.isin([6,7,8,9]))
def corr(c, mask=None):
    a,b = imd[c].values, era5[c].values
    if mask is not None: a,b=a[mask],b[mask]
    return np.corrcoef(a,b)[0,1] if (a.std()>0 and b.std()>0) else np.nan
rows=[]
for c in selected:
    rainy = imd[c].values>1
    rows.append({"station":c.strip(), "full":corr(c), "monsoon":corr(c,mon),
                 "rainy":(corr(c,rainy) if rainy.sum()>30 else np.nan),
                 "acf_imd":imd[c].autocorr(1), "acf_era5":era5[c].autocorr(1)})
cdf = pd.DataFrame(rows)

fig, axs = plt.subplots(2,2, figsize=(13,10))
axs[0,0].boxplot([cdf.full.dropna(), cdf.monsoon.dropna(), cdf.rainy.dropna()], showmeans=True)
axs[0,0].set_xticks([1,2,3]); axs[0,0].set_xticklabels(["full","monsoon","rainy(IMD>1)"])
axs[0,0].set_title("A. Per-station Pearson r (IMD vs ERA5), 16 stations")
axs[0,0].set_ylabel("Pearson r"); axs[0,0].axhline(0.5,ls=":",c="0.6")
for i,s in enumerate(["full","monsoon","rainy"]):
    axs[0,0].text(i+1,0.02,"med="+("%.2f"%cdf[s].median()),ha="center",fontsize=8,color="tab:blue")

bins=np.linspace(0,150,60)
axs[0,1].hist(imd.values.ravel(), bins=bins, alpha=0.55, label="IMD (gauge)", color="tab:orange", log=True)
axs[0,1].hist(era5.values.ravel(), bins=bins, alpha=0.55, label="ERA5", color="tab:blue", log=True)
axs[0,1].set_title("B. Daily rainfall distribution (log count)\nERA5 over-rains on low days, truncates extremes")
axs[0,1].set_xlabel("rainfall (mm/day)"); axs[0,1].set_ylabel("count (log)"); axs[0,1].legend()

x=np.arange(len(cdf))
axs[1,0].bar(x-0.2, cdf.acf_imd, width=0.4, label="IMD (mean "+("%.2f"%cdf.acf_imd.mean())+")", color="tab:orange")
axs[1,0].bar(x+0.2, cdf.acf_era5, width=0.4, label="ERA5 (mean "+("%.2f"%cdf.acf_era5.mean())+")", color="tab:blue")
axs[1,0].set_title("C. Lag-1 autocorrelation per station\nHigher ACF => more predictable day-to-day")
axs[1,0].set_ylabel("lag-1 autocorrelation"); axs[1,0].set_xticks(x)
axs[1,0].set_xticklabels(cdf.station, rotation=90, fontsize=7); axs[1,0].legend()

med_station = cdf.loc[(cdf.full-cdf.full.median()).abs().idxmin(),"station"]
col = [c for c in selected if c.strip()==med_station][0]
sl = slice("2019-06-01","2019-09-30")
axs[1,1].plot(imd.loc[sl].index, imd.loc[sl,col], label="IMD", color="tab:orange", lw=1.1)
axs[1,1].plot(era5.loc[sl].index, era5.loc[sl,col], label="ERA5", color="tab:blue", lw=1.1, alpha=0.85)
axs[1,1].set_title("D. Example station "+med_station+" - monsoon 2019 (r="+("%.2f"%cdf.full.median())+")")
axs[1,1].set_ylabel("rainfall (mm/day)"); axs[1,1].legend(); axs[1,1].tick_params(axis="x",rotation=30)

plt.suptitle("ERA5 vs IMD on the 16 exp_14 stations (2000-2020): same mean, very different variability", fontsize=13, y=1.0)
plt.tight_layout(); plt.savefig(RES + "/era5_vs_imd_characterization.png", dpi=130); plt.show()
print("WROTE era5_vs_imd_characterization.png | median full r=%.3f | acf IMD %.3f vs ERA5 %.3f" % (cdf.full.median(), cdf.acf_imd.mean(), cdf.acf_era5.mean()))
print("example station:", med_station)